<a href="https://colab.research.google.com/github/jatinkumar-creator/ForensicAI/blob/main/notebooks/01_file_signature_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd

print("NumPy Version:", np.__version__)
print("Pandas Version:", pd.__version__)

# NumPy test
numbers = np.array([10, 20, 30, 40, 50])
print("NumPy Array:", numbers)
print("Mean:", np.mean(numbers))

# Pandas test
data = {
    "Evidence_ID": ["EV001", "EV002", "EV003"],
    "File_Type": ["JPEG", "PDF", "SQLite"],
    "Status": ["Pending", "Processed", "Pending"]
}

df = pd.DataFrame(data)

print("\nForensic Evidence Sample:")
display(df)

NumPy Version: 2.4.6
Pandas Version: 2.2.3
NumPy Array: [10 20 30 40 50]
Mean: 30.0

Forensic Evidence Sample:


,Evidence_ID,File_Type,Status
0,EV001,JPEG,Pending
1,EV002,PDF,Processed
2,EV003,SQLite,Pending


In [4]:
#Create synthetic forensic samples

from pathlib import Path

# Create a workspace for our experiment
BASE_DIR = Path("/content/forensicai_lab")
SAMPLE_DIR = BASE_DIR / "samples"

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

# Synthetic files with recognizable file signatures
samples = {
    "image1.jpg": bytes.fromhex("FF D8 FF") + b"SYNTHETIC JPEG DATA",
    "document.pdf": b"%PDF-" + b"SYNTHETIC PDF DATA",
    "image2.png": bytes.fromhex(
        "89 50 4E 47 0D 0A 1A 0A"
    ) + b"SYNTHETIC PNG DATA",
    "database.db": b"SQLite format 3\x00" + b"SYNTHETIC DB DATA",
    "archive.zip": bytes.fromhex("50 4B 03 04") + b"SYNTHETIC ZIP DATA",
    "unknown.bin": b"UNKNOWN FILE CONTENT"
}

# Write samples to disk
for filename, content in samples.items():
    file_path = SAMPLE_DIR / filename
    file_path.write_bytes(content)

print("Synthetic forensic samples created.")

for file_path in SAMPLE_DIR.iterdir():
    print(file_path.name, "-", file_path.stat().st_size, "bytes")

Synthetic forensic samples created.
image2.png - 26 bytes
document.pdf - 23 bytes
unknown.bin - 20 bytes
archive.zip - 22 bytes
database.db - 33 bytes
image1.jpg - 22 bytes


In [5]:
#Build the file identification engine
#Define file signatures

FILE_SIGNATURES = {
    "JPEG": bytes.fromhex("FF D8 FF"),
    "PNG": bytes.fromhex("89 50 4E 47 0D 0A 1A 0A"),
    "PDF": b"%PDF-",
    "ZIP": bytes.fromhex("50 4B 03 04"),
    "SQLite": b"SQLite format 3\x00",
    "GIF": b"GIF87a",
}

In [6]:
#Write the identification function
def identify_file(file_path):

    file_path = Path(file_path)

    # Read only the first 32 bytes
    with file_path.open("rb") as file:
        header = file.read(32)

    # Compare the header with known signatures
    for file_type, signature in FILE_SIGNATURES.items():

        if header.startswith(signature):
            return file_type

    return "Unknown"

In [9]:
#Test the identification engine
#Analyze all synthetic samplesimport pandas as pd

import pandas as pd
results = []

for file_path in SAMPLE_DIR.iterdir():

    detected_type = identify_file(file_path)

    results.append({
        "Filename": file_path.name,
        "Detected Format": detected_type,
        "File Size": file_path.stat().st_size
    })

results_df = pd.DataFrame(results)

display(results_df)

,Filename,Detected Format,File Size
0,image2.png,PNG,26
1,document.pdf,PDF,23
2,unknown.bin,Unknown,20
3,archive.zip,ZIP,22
4,database.db,SQLite,33
5,image1.jpg,JPEG,22


In [8]:
#Test the engine with automated checks
#Validation

expected_results = {
    "image1.jpg": "JPEG",
    "document.pdf": "PDF",
    "image2.png": "PNG",
    "database.db": "SQLite",
    "archive.zip": "ZIP",
    "unknown.bin": "Unknown"
}

for filename, expected_type in expected_results.items():

    actual_type = identify_file(SAMPLE_DIR / filename)

    assert actual_type == expected_type, (
        f"Failed: {filename}. "
        f"Expected {expected_type}, got {actual_type}"
    )

print("All file identification tests passed!")

All file identification tests passed!
